# Representación vectorial del corpus de libros

Este notebook toma el corpus de sinopsis extraído en el TP1 (`libros.csv`) y lo lleva a distintas representaciones vectoriales, siguiendo la progresión de la Unidad 2: primero preparamos dos versiones del texto (una cruda y otra limpia/tokenizada), después entrenamos y comparamos embeddings de palabra (Word2Vec propio vs. un modelo pre-entrenado), y por último generamos embeddings de oración con un modelo tipo SBERT.

Está organizado en tres partes:
- **Parte A** — Preprocesamiento del corpus (limpieza y normalización)
- **Parte B** — Embeddings de palabra: Word2Vec propio vs. modelo pre-entrenado (SBW)
- **Parte C** — Embeddings de oración con Sentence-BERT

# PARTE A

## Qué hacemos en esta celda y para qué

Cargamos el CSV que generamos en el TP1 y preparamos **dos versiones distintas del mismo texto**, porque cada técnica de vectorización que vamos a usar más adelante necesita una entrada distinta:

- `sinopsis_cruda`: solo normalizamos espacios en blanco, sin tocar mayúsculas, puntuación ni stopwords. Esta versión es la que le vamos a dar a **SBERT** en la Parte C, porque ese tipo de modelo fue entrenado sobre texto natural y usa el orden de las palabras y las palabras funcionales (artículos, preposiciones) para entender el significado de la oración. Limpiarlo de más sería tirar información que el modelo sí sabe aprovechar.
- `sinopsis_limpia`: acá sí aplicamos la limpieza clásica de bolsa de palabras (minúsculas, sin puntuación, sin números, sin stopwords en español). Esta es la que le vamos a dar a **Word2Vec** en la Parte B, porque ese modelo trabaja palabra por palabra y no le importa el orden ni la gramática — nos conviene reducir el vocabulario a las palabras que realmente aportan significado.

También descartamos de entrada las filas sin sinopsis, porque no tendría sentido intentar vectorizar un texto vacío.

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('spanish'))

df = pd.read_csv('../data/libros.csv')

# Descartamos las filas sin sinopsis, porque no hay nada que vectorizar en ellas
df = df.dropna(subset=['sinopsis']).reset_index(drop=True)

def limpiar_crudo(texto):
    # Solo colapsamos espacios/saltos de línea repetidos; no tocamos mayúsculas
    # ni puntuación porque esta versión está pensada para un modelo de oración,
    # que necesita el texto lo más parecido posible al original.
    return re.sub(r'\s+', ' ', str(texto)).strip()

def limpiar_tokenizado(texto):
    # Esta versión sí se destruye a propósito: la vamos a usar para un modelo
    # de bolsa de palabras (Word2Vec), que no necesita ni orden ni gramática,
    # solo qué palabras aparecen.
    texto = str(texto).lower()
    texto = re.sub(r'[^\w\s]', ' ', texto)  # sacamos puntuación
    texto = re.sub(r'\d+', ' ', texto)        # sacamos números

    palabras = texto.split()
    palabras_limpias = [p for p in palabras if p not in stop_words]

    return ' '.join(palabras_limpias)

df['sinopsis_cruda'] = df['sinopsis'].apply(limpiar_crudo)
df['sinopsis_limpia'] = df['sinopsis'].apply(limpiar_tokenizado)

print("Muestra del texto crudo (con contexto):")
print(df['sinopsis_cruda'].iloc[0][:150], "...\n")

print("Muestra del texto limpio (bolsa de palabras):")
print(df['sinopsis_limpia'].iloc[0][:150], "...")


# PARTE B

## Qué hacemos en esta celda y para qué

El objetivo de esta parte es comparar dos formas de conseguir embeddings de palabra:

1. **Entrenar nuestro propio Word2Vec** usando únicamente el corpus de 150 libros que armamos en el TP1. Esto nos va a dar un vocabulario chico y vectores que reflejan específicamente el dominio de nuestras sinopsis (literatura clásica), pero con el riesgo de que, al ser tan poco texto, los vectores sean menos estables o menos representativos del lenguaje en general.
2. **Cargar un modelo pre-entrenado (SBW — Spanish Billion Words)**, entrenado sobre miles de millones de palabras de texto en español. Este va a tener un vocabulario mucho más amplio y vectores más "generales", pero no está ajustado a las particularidades de nuestro corpus.

Después comparamos, para un puñado de palabras del dominio (`amor`, `guerra`, `muerte`, `familia`), cuáles son sus vecinos más cercanos en cada modelo. La idea es ver en la práctica cómo cambia la noción de "similitud" entre un modelo entrenado con poquísimo texto específico y uno entrenado con muchísimo texto general.

In [ ]:
from gensim.models import Word2Vec, KeyedVectors
import gdown
import os

# 1. Preparamos el corpus para nuestro modelo (lista de listas de palabras)
oraciones_propias = [texto.split() for texto in df['sinopsis_limpia']]

# 2. Entrenamos nuestro propio Word2Vec
# Parámetros elegidos (a justificar en el informe):
# - vector_size=100: como nuestro corpus es diminuto (150 libros), 300 dimensiones
#   generaría vectores muy ruidosos, con poca evidencia para llenar cada dimensión.
# - window=5: ventana de contexto estándar (5 palabras a la izquierda, 5 a la derecha).
# - min_count=2: descartamos palabras que aparecen una sola vez (hapax) para no
#   generar ruido con términos que no tienen suficiente contexto para aprender un
#   vector confiable.
# - sg=1: usamos Skip-gram, que suele funcionar mejor que CBOW con datasets pequeños.
print("Entrenando nuestro modelo Word2Vec propio...")
modelo_propio = Word2Vec(sentences=oraciones_propias, vector_size=100, window=5, min_count=2, sg=1)

# 3. Descargamos y cargamos el modelo pre-entrenado (Spanish Billion Words)
ruta_modelo_sbw = '../data/SBW-vectors-300-min5.bin.gz'

if not os.path.exists(ruta_modelo_sbw):
    print("\nDescargando el modelo pre-entrenado SBW (esto puede tardar unos minutos)...")
    url_sbw = 'https://drive.google.com/uc?id=1V8hNcnGEyrz0c_dA-v0_5sg31bNgy75r'
    gdown.download(url_sbw, ruta_modelo_sbw, quiet=False)

print("\nCargando el modelo pre-entrenado (requiere bastante RAM)...")
modelo_sbw = KeyedVectors.load_word2vec_format(ruta_modelo_sbw, binary=True)

# 4. Comparamos los vecinos más cercanos de 4 palabras del dominio "Clásicos"
palabras_dominio = ['amor', 'guerra', 'muerte', 'familia']

print("\n--- COMPARACIÓN DE VECINOS MÁS CERCANOS ---")
for palabra in palabras_dominio:
    print(f"\nPalabra objetivo: '{palabra.upper()}'")

    # Vecinos en nuestro modelo propio
    print("Vecinos en nuestro modelo propio:")
    if palabra in modelo_propio.wv:
        vecinos_propios = modelo_propio.wv.most_similar(palabra, topn=3)
        for v, sim in vecinos_propios:
            print(f"  - {v} ({sim:.3f})")
    else:
        print("  (La palabra no existe en el vocabulario de nuestro corpus)")

    # Vecinos en el modelo pre-entrenado
    print("Vecinos en modelo pre-entrenado (SBW):")
    if palabra in modelo_sbw:
        vecinos_sbw = modelo_sbw.most_similar(palabra, topn=3)
        for v, sim in vecinos_sbw:
            print(f"  - {v} ({sim:.3f})")
    else:
        print("  (La palabra no existe en SBW)")


# PARTE C

## Qué hacemos en esta celda y para qué

Acá pasamos de embeddings de **palabra** (Word2Vec) a embeddings de **oración/documento**, usando un modelo de la familia Sentence-BERT (`distiluse-base-multilingual-cased-v1`). A diferencia de Word2Vec, este modelo no nos da un vector por palabra que después tengamos que combinar nosotros — directamente nos devuelve un vector que representa el significado de toda la sinopsis.

Por eso acá sí le pasamos la versión **cruda** del texto (`sinopsis_cruda`) y no la limpia: el modelo fue entrenado sobre oraciones reales, con su puntuación y sus palabras funcionales, así que limpiarlo de antemano solo le sacaría información útil.

Además de generar los embeddings, reportamos dos datos que son importantes para entender las limitaciones del modelo:
- La **dimensión** del embedding que produce (para saber con qué tamaño de vector vamos a trabajar más adelante).
- El **límite de tokens** que el modelo puede procesar de una vez, y cuántas de nuestras sinopsis superan ese límite y por lo tanto se **truncan** silenciosamente (el modelo no avisa cuando esto pasa, hay que calcularlo nosotros).

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Cargamos el modelo pre-entrenado indicado por la cátedra
nombre_modelo = 'distiluse-base-multilingual-cased-v1'
print(f"Cargando modelo {nombre_modelo}...")
modelo_sbert = SentenceTransformer(nombre_modelo)

# 2. Reportamos la dimensión y el límite de tokens del modelo
dimension = modelo_sbert.get_sentence_embedding_dimension()
limite_tokens = modelo_sbert.max_seq_length

print("\n--- REPORTES DEL MODELO ---")
print(f"Dimensión de los embeddings: {dimension}")
print(f"Límite de tokens del modelo: {limite_tokens}")

# 3. Calculamos cuántos documentos se truncan
# Usamos el tokenizador interno del modelo para contar los tokens reales de
# cada sinopsis cruda, sin dejar que el propio modelo la trunque en silencio.
tokenizer = modelo_sbert.tokenizer
truncados = 0

for texto in df['sinopsis_cruda']:
    # Tokenizamos el texto sin truncarlo, para ver su longitud real
    tokens = tokenizer.encode(texto, add_special_tokens=True)
    if len(tokens) > limite_tokens:
        truncados += 1

print(f"Documentos truncados por superar el límite: {truncados} de {len(df)}")

# 4. Generamos los embeddings para todo el corpus
print("\nGenerando embeddings de documentos (esto puede tardar unos segundos)...")
# Le pasamos la sinopsis CRUDA (con puntuación y stopwords) a propósito, ya que
# este modelo fue entrenado sobre texto natural y usa esa información.
embeddings_sbert = modelo_sbert.encode(df['sinopsis_cruda'].tolist(), show_progress_bar=True)

print("Embeddings generados exitosamente. Forma de la matriz:", embeddings_sbert.shape)
